# 🥥 Janjang Vision - Training Penuh (GPU Colab)

Notebook ini melatih model deteksi janjang kelapa sawit (TBS) dengan **YOLO11n** di GPU gratis Colab.

**Persiapan (di laptop kamu):**
1. Buka <https://colab.research.google.com> dan login akun Google
2. Menu **Runtime -> Change runtime type -> Hardware accelerator: T4 GPU**
3. Siapkan 2 file dari folder `D:\Project\janjang-vision`:
   - `dataset_janjang_full.zip` (dataset 850 gambar)
   - `janjang_counter.py` (script project)

**Alur:** install library → upload file → jalankan training (±30-60 menit) → download model `best.pt`.

Jalankan sel satu per satu dari atas ke bawah (klik tombol ▶ di kiri atas tiap sel).

In [ ]:
# 1) Install library Ultralytics (sekali saja per sesi)
!pip install -q ultralytics
import ultralytics
print('Ultralytics', ultralytics.__version__, '- OK')

## 2) Upload file

Klik ikon folder di kiri, atau jalankan sel di bawah lalu pilih file. **Pilih 2 file sekaligus**: `dataset_janjang_full.zip` dan `janjang_counter.py` (tahan Ctrl/Cmd saat klik).

In [ ]:
from google.colab import files
uploaded = files.upload()
print('Terunggah:', list(uploaded.keys()))

**Alternatif (kalau file besar/upload lambat):** pakai Google Drive — upload zip ke Drive, lalu jalankan sel di bawah (ganti `NAMA_FILE.zip` dengan nama file kamu). Kalau sudah pakai cara upload biasa, lewati sel ini.

In [ ]:
# (OPSIONAL) Alternatif: ambil dari Google Drive
from google.colab import drive
drive.mount('/content/drive')
!cp "/content/drive/MyDrive/NAMA_FILE.zip" ./dataset_janjang_full.zip  # ganti nama file
!cp "/content/drive/MyDrive/janjang_counter.py" ./janjang_counter.py

In [ ]:
# 3) Ekstrak dataset v2 & perbaiki path di dataset.yaml# PENTING: hanya memproses dataset_janjang_v2.zip (hapus zip lama di Colab dulu!)import glob, os, re, zipfile, shutilZIP_NAME = 'dataset_janjang_v2.zip'# bersihkan sisa ekstrak lamafor name in os.listdir('.'):    if '\\' in name or name == 'dataset':        p = os.path.join('.', name)        if os.path.isdir(p):            shutil.rmtree(p, ignore_errors=True)        else:            os.remove(p)if not os.path.exists(ZIP_NAME):    raise SystemExit('ZIP v2 tidak ditemukan! Upload dataset_janjang_v2.zip dulu.')with zipfile.ZipFile(ZIP_NAME) as zf:    for info in zf.infolist():        target = info.filename.replace('\\', '/')        target = os.path.join('.', target)        if info.is_dir() or info.filename.endswith('/'):            os.makedirs(target, exist_ok=True)            continue        os.makedirs(os.path.dirname(target), exist_ok=True)        if os.path.isdir(target):            continue        with zf.open(info) as src, open(target, 'wb') as dst:            dst.write(src.read())print('Zip v2 diekstrak:', ZIP_NAME)# perbaiki path di dataset.yamlyaml_list = glob.glob('**/dataset.yaml', recursive=True)print('dataset.yaml di:', yaml_list)if yaml_list:    p = yaml_list[0]    base = '/content/' + os.path.dirname(p).replace('\\', '/').strip('/')    txt = open(p).read()    txt = re.sub(r'^path:.*$', f'path: {base}', txt, flags=re.M)    open(p, 'w').write(txt)    print('path diset ke:', base)print('Script ada?      :', os.path.exists('janjang_counter.py'))print('Jumlah gambar    :', len(glob.glob('dataset/images/train/*.jpg')))

## 4) Training 🚀

100 epoch, ukuran 640px, early-stop otomatis jika tidak ada peningkatan (patience 25).
Estimasi waktu di GPU T4 gratis: **±30-60 menit** (bisa lebih lama saat Colab sedang ramai).

> ⚠️ Biarkan tab ini terbuka. Kalau koneksi putus, Colab biasanya menyimpan progress otomatis.

In [ ]:
!python janjang_counter.py --train dataset/dataset.yaml --epochs 100 --imgsz 640 --batch 16 --patience 25

## 5) Download model hasil training

File `best.pt` akan terunduh ke folder Download laptop kamu.

In [ ]:
from google.colab import files
import glob

best = sorted(glob.glob('runs/detect/*/weights/best.pt'))
print('Model ditemukan:', best)
if best:
    files.download(best[-1])

## 6) Uji coba di Colab (opsional)

Upload satu foto janjang, lalu jalankan sel di bawah untuk melihat hasil deteksi langsung.

In [ ]:
from google.colab import files
files.upload()  # pilih foto uji (mis. foto_uji.png)
import glob
best = sorted(glob.glob('runs/detect/*/weights/best.pt'))[-1]
!python janjang_counter.py foto_uji.png --model {best} --conf 0.3
from IPython.display import Image
Image('foto_uji_hasil.png')

## 7) Setelah selesai — pakai model di laptop

1. Pindahkan `best.pt` (hasil download tadi) ke folder `D:\Project\janjang-vision`
2. Jalankan di Command Prompt / PowerShell:

```
cd D:\Project\janjang-vision
python janjang_counter.py foto.jpg --model best.pt
```

Model hasil Colab ini jauh lebih baik dari versi CPU karena dilatih di GPU dengan 100 epoch penuh.